[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QuantLet/NN_DL/blob/main/NN_DL_LLM_Workshop/NN_DL_LLM_NanoGPT/NN_DL_LLM_NanoGPT.ipynb)

# Building a Large Language Model from Scratch

**NanoGPT: A minimal GPT built from scratch in ~300 lines of PyTorch**

**Author:** Daniel Traian Pele, ASE Bucharest / IDA

**Keywords:** GPT, transformer, attention, language model, nanoGPT, from scratch


In [ ]:
!pip install torch matplotlib numpy -q


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import urllib.request
import os

torch.manual_seed(42)
np.random.seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

# --- Course color palette ---
MAIN_BLUE = '#003DA5'
IDA_RED   = '#C8102E'
FOREST    = '#228B22'
CRIMSON   = '#DC143C'

plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'figure.figsize': (10, 5.5),
    'savefig.bbox': 'tight',
    'savefig.dpi': 300,
})

FIG_DIR = 'figs'
os.makedirs(FIG_DIR, exist_ok=True)

def save_fig(name, fig=None):
    if fig is None: fig = plt.gcf()
    fig.savefig(os.path.join(FIG_DIR, f'{name}.pdf'), bbox_inches='tight', dpi=300)
    fig.savefig(os.path.join(FIG_DIR, f'{name}.png'), bbox_inches='tight', dpi=300)
    print(f'Saved {FIG_DIR}/{name}.pdf')


## 1. Download Shakespeare Corpus


In [ ]:
url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
urllib.request.urlretrieve(url, 'input.txt')
with open('input.txt', 'r') as f:
    text = f.read()
print(f'Corpus: {len(text):,} characters')
print(f'First 200 chars:\n{text[:200]}')


## 2. Character-Level Tokenizer


In [ ]:
chars = sorted(set(text))
vocab_size = len(chars)
print(f'Vocab size: {vocab_size}')
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])
print(encode('hello'), '->', decode(encode('hello')))


## 3. Data Pipeline


In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

def get_batch(split, batch_size=32, block_size=128):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i+block_size] for i in ix])
    y = torch.stack([d[i+1:i+1+block_size] for i in ix])
    return x.to(device), y.to(device)

print(f'Train: {len(train_data):,} tokens, Val: {len(val_data):,} tokens')


## 4. Token + Positional Embeddings


In [ ]:
class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model, max_seq_len):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
    def forward(self, x):
        B, T = x.shape
        return self.tok_emb(x) + self.pos_emb(torch.arange(T, device=x.device))


## 5. Self-Attention (Single Head)


In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, d_model, head_dim):
        super().__init__()
        self.W_q = nn.Linear(d_model, head_dim, bias=False)
        self.W_k = nn.Linear(d_model, head_dim, bias=False)
        self.W_v = nn.Linear(d_model, head_dim, bias=False)
        self.head_dim = head_dim
    def forward(self, x, mask=None):
        Q, K, V = self.W_q(x), self.W_k(x), self.W_v(x)
        scores = Q @ K.transpose(-2, -1) / (self.head_dim ** 0.5)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        weights = F.softmax(scores, dim=-1)
        return weights @ V, weights


## 6. Multi-Head Attention


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.head_dim = d_model // n_heads
        self.heads = nn.ModuleList([SelfAttention(d_model, self.head_dim) for _ in range(n_heads)])
        self.proj = nn.Linear(d_model, d_model)
    def forward(self, x, mask=None):
        out = torch.cat([h(x, mask)[0] for h in self.heads], dim=-1)
        return self.proj(out)


## 7. Transformer Block


In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
    def forward(self, x, mask=None):
        x = x + self.attn(self.ln1(x), mask)
        x = x + self.ff(self.ln2(x))
        return x


## 8. Complete NanoGPT Model


In [ ]:
class NanoGPT(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, d_ff, max_seq_len):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.embedding = TokenEmbedding(vocab_size, d_model, max_seq_len)
        self.blocks = nn.ModuleList([TransformerBlock(d_model, n_heads, d_ff) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)
        self.register_buffer('mask', torch.tril(torch.ones(max_seq_len, max_seq_len)))
    def forward(self, x, targets=None):
        B, T = x.shape
        h = self.embedding(x)
        mask = self.mask[:T, :T]
        for block in self.blocks:
            h = block(h, mask)
        logits = self.head(self.ln_f(h))
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1)) if targets is not None else None
        return logits, loss
    @torch.no_grad()
    def generate(self, x, max_new_tokens, temperature=1.0):
        for _ in range(max_new_tokens):
            x_cond = x[:, -self.max_seq_len:]
            logits, _ = self(x_cond)
            probs = F.softmax(logits[:, -1, :] / temperature, dim=-1)
            x = torch.cat([x, torch.multinomial(probs, 1)], dim=1)
        return x

model = NanoGPT(vocab_size, d_model=64, n_heads=4, n_layers=4, d_ff=256, max_seq_len=128).to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')
print('Before training:', decode(model.generate(torch.zeros((1,1), dtype=torch.long, device=device), 100)[0].tolist()))


## 9. Training Loop


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
train_losses, val_losses = [], []
sample_steps, sample_texts = [], []

for step in range(5000):
    model.train()
    xb, yb = get_batch('train')
    _, loss = model(xb, yb)
    optimizer.zero_grad(); loss.backward(); optimizer.step()

    if step % 250 == 0:
        model.eval()
        with torch.no_grad():
            xv, yv = get_batch('val')
            _, vl = model(xv, yv)
        train_losses.append(loss.item())
        val_losses.append(vl.item())

        prompt = torch.zeros((1, 1), dtype=torch.long, device=device)
        sample = decode(model.generate(prompt, max_new_tokens=100, temperature=0.8)[0].tolist())
        sample_steps.append(step)
        sample_texts.append(sample)

        print(f'Step {step:5d} | train {loss.item():.3f} | val {vl.item():.3f}')
        print(f'  -> {sample[:80]}...\n')


## 10. Training Curve


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
steps = np.arange(len(train_losses)) * 250
ax.plot(steps, train_losses, color=MAIN_BLUE, lw=2, label='Train')
ax.plot(steps, val_losses, color=IDA_RED, lw=2, label='Val')
ax.set_xlabel('Step')
ax.set_ylabel('Cross-Entropy Loss')
ax.set_title('NanoGPT Training Curve', fontweight='bold')
ax.legend()
plt.tight_layout()
save_fig('nn_llm_training_curve')
plt.show()


## 11. Generate Text at Different Temperatures


In [ ]:
model.eval()
prompt = torch.zeros((1, 1), dtype=torch.long, device=device)
for temp in [0.5, 0.8, 1.0, 1.5]:
    print(f'\n{"="*60}\nTemperature = {temp}\n{"="*60}')
    out = model.generate(prompt, max_new_tokens=200, temperature=temp)
    print(decode(out[0].tolist()))


## 11. Evolution: From Random Noise to Shakespeare


In [ ]:
print("=" * 70)
print("EVOLUTION OF TEXT GENERATION DURING TRAINING")
print("=" * 70)
for i in [0, 4, 8, 12, 16, -1]:
    if abs(i) < len(sample_steps):
        step = sample_steps[i]
        loss = train_losses[i]
        print(f'\n{chr(9472) * 70}')
        print(f'Step {step:,} | Loss: {loss:.3f}')
        print(f'{chr(9472) * 70}')
        print(sample_texts[i][:200])

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
snapshots = [
    (0, f'Step {sample_steps[0]} (untrained)'),
    (len(sample_steps)//2, f'Step {sample_steps[len(sample_steps)//2]:,}'),
    (-1, f'Step {sample_steps[-1]:,} (trained)')
]
for ax, (idx, title) in zip(axes, snapshots):
    txt = sample_texts[idx][:160].replace('\n', chr(8629)+'\n')
    ax.text(0.05, 0.95, txt, transform=ax.transAxes,
            fontsize=8, verticalalignment='top', fontfamily='monospace',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))
    ax.set_title(title, fontweight='bold', fontsize=11)
    ax.axis('off')
fig.suptitle('NanoGPT: From Random Noise to Coherent Text',
             fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
save_fig('nn_llm_text_evolution')
plt.show()


## 13. Interactive Text Generation


In [ ]:
def generate_from_prompt(prompt_text, max_tokens=200, temperature=0.8):
    model.eval()
    valid_chars = [c for c in prompt_text if c in stoi]
    tokens = torch.tensor([encode(''.join(valid_chars))], dtype=torch.long, device=device)
    output = model.generate(tokens, max_new_tokens=max_tokens, temperature=temperature)
    full = decode(output[0].tolist())
    prompt_len = len(''.join(valid_chars))
    print(f'PROMPT: {full[:prompt_len]}')
    print(f'OUTPUT: {full[prompt_len:]}')

generate_from_prompt("ROMEO:\nO, ")
print("\n" + "=" * 60)
generate_from_prompt("The king shall ")
print("\n" + "=" * 60)
generate_from_prompt("To be or not to be")


## 14. What Does the Model Predict Next?


In [ ]:
def plot_next_token_probs(prompt_text, top_k=20):
    model.eval()
    tokens = torch.tensor([encode(prompt_text)], device=device)
    with torch.no_grad():
        logits, _ = model(tokens)
    probs = F.softmax(logits[0, -1, :], dim=-1).cpu().numpy()
    top_idx = probs.argsort()[-top_k:][::-1]
    top_chars = [repr(itos[i]) for i in top_idx]
    top_probs = probs[top_idx]
    fig, ax = plt.subplots(figsize=(10, 6))
    colors = [IDA_RED if i == 0 else MAIN_BLUE for i in range(top_k)]
    ax.barh(range(top_k), top_probs[::-1], color=colors[::-1], edgecolor='white')
    ax.set_yticks(range(top_k))
    ax.set_yticklabels(top_chars[::-1], fontfamily='monospace')
    ax.set_xlabel('Probability')
    ax.set_title(f'Next token after: "{prompt_text}"', fontweight='bold')
    plt.tight_layout()
    return fig

fig = plot_next_token_probs("To be or not to b")
save_fig('nn_llm_next_token_tobe')
plt.show()

fig = plot_next_token_probs("ROMEO:\n")
save_fig('nn_llm_next_token_romeo')
plt.show()


## 15. What Does the Model Attend To?


In [ ]:
def get_attention_weights(model, text_input):
    model.eval()
    tokens = torch.tensor([encode(text_input)], device=device)
    T = tokens.shape[1]
    mask = model.mask[:T, :T]
    with torch.no_grad():
        h = model.embedding(tokens)
        h_norm = model.blocks[0].ln1(h)
        head_weights = []
        for head in model.blocks[0].attn.heads:
            Q = head.q(h_norm)
            K = head.k(h_norm)
            scores = Q @ K.transpose(-2,-1) / head.scale
            scores = scores.masked_fill(mask == 0, float('-inf'))
            w = F.softmax(scores, dim=-1)
            head_weights.append(w[0].cpu().numpy())
    return head_weights, list(text_input)

text_probe = "The king"
weights, char_labels = get_attention_weights(model, text_probe)
n_heads = len(weights)
fig, axes = plt.subplots(1, n_heads, figsize=(4*n_heads, 4))
for i, ax in enumerate(axes):
    ax.imshow(weights[i], cmap='Blues', aspect='auto', vmin=0, vmax=1)
    ax.set_xticks(range(len(char_labels)))
    ax.set_xticklabels(char_labels, fontfamily='monospace', fontsize=9)
    ax.set_yticks(range(len(char_labels)))
    ax.set_yticklabels(char_labels, fontfamily='monospace', fontsize=9)
    ax.set_title(f'Head {i+1}', fontweight='bold')
fig.suptitle(f'Attention: "{text_probe}" (Layer 1)', fontweight='bold', fontsize=13, y=1.02)
plt.tight_layout()
save_fig('nn_llm_attention_heads')
plt.show()


## 16. Where Are the Parameters?


In [ ]:
comps = {
    'Token emb': sum(p.numel() for p in model.embedding.tok_emb.parameters()),
    'Pos emb': sum(p.numel() for p in model.embedding.pos_emb.parameters()),
    'Attention': sum(p.numel() for b in model.blocks for p in b.attn.parameters()),
    'FeedForward': sum(p.numel() for b in model.blocks for p in b.ff.parameters()),
    'LayerNorm': sum(p.numel() for b in model.blocks for ln in [b.ln1,b.ln2] for p in ln.parameters()) + sum(p.numel() for p in model.ln_f.parameters()),
    'Output head': sum(p.numel() for p in model.head.parameters()),
}
total = sum(comps.values())
fig, ax = plt.subplots(figsize=(10, 5))
colors_list = [MAIN_BLUE, '#4169E1', IDA_RED, CRIMSON, FOREST, '#808080']
bars = ax.barh(list(comps.keys()), list(comps.values()), color=colors_list, edgecolor='white')
for bar, val in zip(bars, comps.values()):
    ax.text(bar.get_width() + total*0.01, bar.get_y() + bar.get_height()/2,
            f'{val:,} ({val/total*100:.1f}%)', va='center', fontsize=9)
ax.set_xlabel('Parameters')
ax.set_title(f'NanoGPT Parameters (Total: {total:,})', fontweight='bold')
plt.tight_layout()
save_fig('nn_llm_param_breakdown')
plt.show()


## 17. From NanoGPT to GPT-4


In [ ]:
names = ['Our NanoGPT', 'GPT-2 Small', 'GPT-2 Large', 'GPT-3', 'GPT-4 (est.)']
params = [sum(p.numel() for p in model.parameters()), 124_000_000, 774_000_000, 175_000_000_000, 1_700_000_000_000]
fig, ax = plt.subplots(figsize=(10, 5))
colors = [IDA_RED] + [MAIN_BLUE]*4
bars = ax.barh(names, params, color=colors, edgecolor='white', log=True)
for bar, p in zip(bars, params):
    label = f'{p/1e9:.0f}B' if p >= 1e9 else (f'{p/1e6:.0f}M' if p >= 1e6 else f'{p/1e3:.0f}K')
    ax.text(bar.get_width() * 1.5, bar.get_y() + bar.get_height()/2, label, va='center', fontweight='bold')
ax.set_xlabel('Parameters (log scale)')
ax.set_title('Same Architecture, Different Scale', fontweight='bold')
plt.tight_layout()
save_fig('nn_llm_scaling_comparison')
plt.show()
print(f'GPT-4 is ~{1_700_000_000_000 // sum(p.numel() for p in model.parameters()):,}x our model.')


## Summary

**What we built:** A complete GPT-style language model from scratch in ~300 lines of PyTorch.

**Components:** Token embeddings -> Positional embeddings -> Multi-head causal self-attention -> FeedForward with GELU -> Residual connections -> LayerNorm -> Autoregressive generation

**Key insight:** The architecture of our 0.5M parameter model is *identical* to GPT-4's 1.7 trillion parameter model. The difference is scale.

**Connection to finance:** The same architecture powers LLM-VaR (Pele, 2025), FinGPT, BloombergGPT, and every LLM-based financial application.

---

**Course:** Neural Networks and Deep Learning with Business Applications

**Author:** Daniel Traian Pele, ASE Bucharest / IDA

**Repository:** [QuantLet/NN_DL](https://github.com/QuantLet/NN_DL)
